In [ ]:
!pip install -q timm kagglehub


In [ ]:
import os, gc, math, random, warnings, urllib.request
from pathlib import Path
from io import BytesIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import deque

from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    f1_score, roc_auc_score, roc_curve
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.utils.tensorboard import SummaryWriter
import torchvision.transforms as T

import timm

warnings.filterwarnings("ignore")

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"timm     : {timm.__version__}")


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
os.makedirs('/root/.config/kaggle', exist_ok=True)
os.system('cp /content/drive/MyDrive/AML/kaggle.json /root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print("✓ Kaggle credentials")

import kagglehub
dl_path = kagglehub.dataset_download("cjinny/mrnet-v1")
ROOT = Path(dl_path) / "MRNet-v1.0"
CFG.ROOT = ROOT
assert ROOT.exists(), f"ROOT not found: {ROOT}"
print(f"✓ Dataset ROOT: {ROOT}")


In [ ]:
def load_labels(root, split):
    acl      = pd.read_csv(root/f"{split}-acl.csv",      header=None, names=["id","acl"])
    abnormal = pd.read_csv(root/f"{split}-abnormal.csv", header=None, names=["id","abnormal"])
    meniscus = pd.read_csv(root/f"{split}-meniscus.csv", header=None, names=["id","meniscus"])
    df = acl.merge(abnormal, on="id").merge(meniscus, on="id")
    df["id"] = df["id"].astype(str).str.zfill(4)
    df["split"] = split
    for plane in ["sagittal","coronal","axial"]:
        df[f"{plane}_path"] = df["id"].apply(lambda x: str(root/split/plane/f"{x}.npy"))
    df["target"] = df.apply(
        lambda r: [float(r["acl"]), float(r["meniscus"]), float(r["abnormal"])], axis=1)
    return df

train_df = load_labels(CFG.ROOT, "train")
valid_df = load_labels(CFG.ROOT, "valid")
print(f"Train: {len(train_df)}   Valid: {len(valid_df)}")
for lbl in CFG.LABEL_NAMES:
    n = int(train_df[lbl].sum())
    print(f"  {lbl:>10}: {n} pos ({100*n/len(train_df):.1f}%)  {len(train_df)-n} neg")
